# Project: Q&A Chatbot

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will build a conversational chatbot with memory that can use tools and remember previous messages across turns.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define the Weather Tool

Create a tool the chatbot can use to check weather conditions.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use this when the user asks about weather conditions."""
    weather_data = {
        "new york": "65°F, partly cloudy",
        "london": "55°F, rainy",
        "tokyo": "78°F, sunny",
        "paris": "62°F, overcast",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

## 4. Create an Agent with Memory

Use `InMemorySaver` as a checkpointer to persist conversation state across turns.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("gpt-4o-mini", model_provider="openai")
memory = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=[get_weather],
    prompt="You are a friendly assistant. You can check the weather for users. Be conversational and remember what the user has told you.",
    checkpointer=memory,
)

## 5. Multi-Turn Conversation

Use a `thread_id` in the config to maintain conversation context.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "user-1"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alice.")]},
    config=config,
)
print("Bot:", result["messages"][-1].content)

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather in Tokyo?")]},
    config=config,
)
print("Bot:", result["messages"][-1].content)

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Do you remember my name?")]},
    config=config,
)
print("Bot:", result["messages"][-1].content)

## 6. Separate Threads

Different `thread_id` values create independent conversations with no shared memory.

In [ ]:
config_2 = {"configurable": {"thread_id": "user-2"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather in London?")]},
    config=config_2,
)
print("Bot:", result["messages"][-1].content)

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Do you know my name?")]},
    config=config_2,
)
print("Bot:", result["messages"][-1].content)

## 7. Full Chatbot Code

Here's the complete chatbot in one cell.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use this when the user asks about weather conditions."""
    weather_data = {
        "new york": "65°F, partly cloudy",
        "london": "55°F, rainy",
        "tokyo": "78°F, sunny",
        "paris": "62°F, overcast",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

model = init_chat_model("gpt-4o-mini", model_provider="openai")
memory = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=[get_weather],
    prompt="You are a friendly assistant. You can check the weather for users. Be conversational and remember what the user has told you.",
    checkpointer=memory,
)

config = {"configurable": {"thread_id": "session-1"}}

questions = [
    "Hi! My name is Alice.",
    "What's the weather like in Tokyo?",
    "How about Paris?",
    "What was the first city I asked about?",
]

for question in questions:
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,
    )
    print(f"User: {question}")
    print(f"Bot: {result['messages'][-1].content}")
    print()

## Summary

- `InMemorySaver` provides conversation memory by checkpointing agent state
- `thread_id` groups messages into a conversation thread
- Different thread IDs create independent, isolated conversations
- Combining `create_react_agent` with a checkpointer creates a stateful, tool-using chatbot